# 📸 debim: 3D Visual Regression & Component-Level Benchmark

[![GitHub Repository](https://img.shields.io/badge/GitHub-PRIDA--TAKON%2Fdebim-181717?logo=github&style=for-the-badge)](https://github.com/PRIDA-TAKON/debim)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg?style=for-the-badge)](https://github.com/PRIDA-TAKON/debim/blob/main/LICENSE)

> **debim** (Declarative BIM) Component-Level 3D Visual Regression Suite.
> Evaluates geometric fidelity between Original IFC models vs debim's Declarative YAML & 3D Viewer representations.
>
> Prevents false-positive conversions where YAML is syntactically valid but 3D geometry is misoriented or displaced.

In [ ]:
# 1. Install dependencies & clone latest debim main branch
!pip install -q ifcopenshell "pydantic>=2.0" pyyaml rich trimesh matplotlib pillow pandas tabulate

!rm -rf /kaggle/working/debim
!git clone https://github.com/PRIDA-TAKON/debim.git /kaggle/working/debim

import sys
sys.path.insert(0, "/kaggle/working/debim/src")
sys.path.insert(0, "/kaggle/working/debim")
print("Environment initialized with latest debim commit from GitHub.")

In [ ]:
# 2. Discover IFC files across all mounted datasets
from pathlib import Path

input_root = Path("/kaggle/input")
all_ifc_files = sorted(list(input_root.glob("**/*.ifc")))
print(f"Found {len(all_ifc_files)} IFC files across datasets.")

# Filter out 0-byte or corrupted files and pick diverse samples
valid_files = [f for f in all_ifc_files if f.stat().st_size > 5000]
print(f"Valid non-empty IFC files: {len(valid_files)}")
for idx, f in enumerate(valid_files[:10], 1):
    print(f"  [{idx}] {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 3. Execute 3D Visual Regression Comparator across Sampled Models
import os
from tools.render_visual_regression import run_visual_regression

output_dir = Path("/kaggle/working/visual_regression_output")
output_dir.mkdir(parents=True, exist_ok=True)

# Sample up to 12 representative models across sizes
test_sample = valid_files[:12]
summary_stats = []

for idx, ifc_p in enumerate(test_sample, 1):
    print(f"\n[{idx}/{len(test_sample)}] Evaluating: {ifc_p.name}")
    model_out = output_dir / ifc_p.stem
    try:
        stat = run_visual_regression(
            ifc_path=ifc_p,
            output_dir=model_out,
            max_elements_per_type=2,
            limit_total=8,
        )
        summary_stats.append({"name": ifc_p.name, **stat})
    except Exception as e:
        print(f"  [SKIP] Error processing {ifc_p.name}: {e}")

print("\nVisual regression execution complete.")

In [ ]:
# 4. Display KPI Summary & Visual Diff Gallery
import pandas as pd
from IPython.display import display, HTML, Markdown

if summary_stats:
    df = pd.DataFrame(summary_stats)
    total_tested = df["total_tested"].sum()
    total_passed = df["pass_count"].sum()
    overall_pass_rate = (total_passed / total_tested * 100) if total_tested else 0.0
    mean_match = df["avg_match"].mean()

    display(Markdown(f"""
### 📊 Overall 3D Visual Regression KPIs
- **Total Models Evaluated:** {len(df)}
- **Total 3D Components Tested:** {total_tested}
- **Overall Component Pass Rate (>=85%):** {total_passed}/{total_tested} ({overall_pass_rate:.1f}%)
- **Mean Visual Fidelity Match:** {mean_match:.1f}%
"""))
    display(df[["name", "total_tested", "pass_count", "avg_match"]])

    # Gallery of sample diff images
    all_images = list(output_dir.glob("**/*.png"))
    print(f"\nGenerated {len(all_images)} 3D component diff images.")
    display(Markdown("### 🖼️ Sample 3D Visual Diff Gallery (Original | debim | Diff Map)"))
    for img_p in all_images[:6]:
        display(Markdown(f"**{img_p.parent.name} - {img_p.stem}**"))
        display(Image(filename=str(img_p)))
else:
    print("No models evaluated.")